OUTLIER DETECTION USING Z-SCORE AND INTERQUARTILE RANGE

In [26]:
import mysql.connector
import configparser
import pandas as pd
from scipy.stats import iqr
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\erfpPROD.ini')

In [27]:
host=config['erfpPROD']['host']
user=config['erfpPROD']['user']
pwd=config['erfpPROD']['pwd']
database=config['erfpPROD']['database']

In [28]:
conn = mysql.connector.connect(
          host=host,
          user=user,
          passwd=pwd,
          database=database)
cursor = conn.cursor()

In [29]:
# CSR_CARBON
import_query_location = 'C:\\Users\\USER\\Documents\\Green_score\\NORMALISED\\CSR_CARBON.sql' 
import_query_open = open(import_query_location, 'r', encoding="utf8")
import_query_read = import_query_open.read()

# CSR_CARBON_CERFITIED
import_query_location_1 = 'C:\\Users\\USER\\Documents\\Green_score\\NORMALISED\\CSR_CARBON_CERTIFIED.sql' 
import_query_open_1 = open(import_query_location_1, 'r', encoding="utf8")
import_query_read_1 = import_query_open_1.read()

In [34]:
cursor.execute(import_query_read)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df.columns = df_col_names
erfp_df.MEAS_CARBON_AMT_LBS = pd.to_numeric(erfp_df.MEAS_CARBON_AMT_LBS)
erfp_df.head()

In [35]:
#####################################################################
cursor.execute(import_query_read_1)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df_1 = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df_1 = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df_1.columns = df_col_names
erfp_df_1.MEAS_CARBON_AMT_LBS = pd.to_numeric(erfp_df_1.MEAS_CARBON_AMT_LBS)
erfp_df_1.head()

In [14]:
erfp_df.dtypes,erfp_df_1.dtypes

In [10]:
MEAS_ENERGY_AMT = erfp_df.MEAS_ENERGY_AMT.to_numpy()
#iqr(MEAS_ENERGY_AMT)
MEAS_ENERGY_AMT

In [20]:
# Visualising to FIND the outliers

i = 'MEAS_CARBON_AMT_LBS'
 
plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df_1[i].min()*1.5, erfp_df_1[i].max()*1.1)
plt.title("Kernel Distribution - Carbon amount (line graph)") 
ax = erfp_df_1[i].plot(kind='kde').margins(0.1)
 
plt.subplot(212)
plt.xlim(erfp_df_1[i].min()*1.5, erfp_df_1[i].max()*1.1)
plt.title("Kernel Distribution - Carbon amount (box plot)") 
sns.boxplot(y=erfp_df_1[i])

In [36]:
# Remove any zeros (otherwise we get (-inf)
erfp_df_1.loc[erfp_df_1.MEAS_CARBON_AMT_LBS == 0, 'MEAS_CARBON_AMT_LBS'] = np.nan
erfp_df_1.loc[erfp_df_1.MEAS_CARBON_AMT_LBS == 9999, 'MEAS_CARBON_AMT_LBS'] = np.nan
erfp_df_1.loc[erfp_df_1.MEAS_CARBON_AMT_LBS == 9999.99, 'MEAS_CARBON_AMT_LBS'] = np.nan
 
# Drop NA
erfp_df_1.dropna(inplace=True)

# Remove any zeros (otherwise we get (-inf)
erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 0, 'MEAS_CARBON_AMT_LBS'] = np.nan
erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 9999, 'MEAS_CARBON_AMT_LBS'] = np.nan
erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 9999.99, 'MEAS_CARBON_AMT_LBS'] = np.nan
 
# Drop NA
erfp_df.dropna(inplace=True)

In [88]:
# Visualising Log Transform
i = 'MEAS_CARBON_AMT_LBS'
erfp_df_1['Log_' + i] = np.log(erfp_df_1[i])
erfp_df['Log_' + i] = np.log(erfp_df[i])

i = 'Log_MEAS_CARBON_AMT_LBS'

all_hotels = erfp_df[i].tolist()
certified_hotels = erfp_df_1[i].tolist()
data = [all_hotels, certified_hotels]

plt.figure(figsize=(16,8))
plt.subplot(211)
plt.xlim(erfp_df_1[i].min()*1.1, erfp_df_1[i].max()*1.1)
# ax = erfp_df_1[i].plot.kde()
# sns.distplot(erfp_df_1[i])
sns.kdeplot(erfp_df_1[i], label = 'CERTIFIED_HOTELS', shade = True)
sns.kdeplot(erfp_df[i], label = 'ALL_HOTELS', shade = True)
plt.title("Kernel Distribution of Log transfomed values - Carbon amount (line graph)") 


plt.figure(figsize=(16,8)) 
plt.subplot(212)
plt.xlim(erfp_df_1[i].min()*1.1, erfp_df_1[i].max()*1.1)
sns.boxplot(x=erfp_df_1[i])
plt.title("Kernel Distribution of Log transfomed values - Carbon amount (box graph)")

In [96]:
erfp_df['MEAS_CARBON_AMT_LBS'].describe()

In [97]:
i = 'Log_MEAS_CARBON_AMT_LBS'

all_hotels = erfp_df[i].tolist()
certified_hotels = erfp_df_1[i].tolist()

data = [all_hotels, certified_hotels]
labels=['ALL HOTELS', 'CERTIFIED HOTELS']
plt.subplots(figsize=(10,8))
plt.ylim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
sns.boxplot(data=data, notch='True')
plt.xticks(np.arange(0,2), labels)
plt.xlabel('CARBON AMOUNT DATA LOG TRANSFORMED') 
plt.ylabel('LOG VALUES') 
plt.yticks(np.arange(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1, 1.0))
plt.title("Kernel Distribution of Log transfomed values - Carbon amount (box graph)")

In [92]:
# OUTLIER DETECTION USING Z -SCORE
i = 'MEAS_CARBON_AMT_LBS'
_mean = erfp_df_1[i].mean()
_std = erfp_df_1[i].std()

erfp_df_1[i + '_Zscore'] = (erfp_df_1[i] - _mean)/(_std)
erfp_df_1[i + '_Zscore_abs'] = erfp_df_1[i + '_Zscore'].apply(np.abs)


In [93]:
i = 'MEAS_CARBON_AMT_LBS'
_mean = erfp_df[i].mean()
_std = erfp_df[i].std()

erfp_df[i + '_Zscore'] = (erfp_df[i] - _mean)/(_std)
erfp_df[i + '_Zscore_abs'] = erfp_df[i + '_Zscore'].apply(np.abs)

In [52]:
erfp_df[i + '_Zscore'].hist(color='slategray')
plt.title("Standard Normal Distribution fo Z-Score - Carbon amount", y=1.015, fontsize=22)
plt.xlabel("z-score", labelpad=14)
plt.ylabel(i, labelpad=14);

In [54]:
erfp_df['MEAS_CARBON_AMT_LBS_Zscore_abs'].min()

In [57]:
# erfp_df['MEAS_CARBON_AMT_LBS_Zscore_abs']
# # Visualising Log Transform
# erfp_df['Log_' + i] = np.log(erfp_df[i])

i = 'MEAS_CARBON_AMT_LBS_Zscore_abs'
 
plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
 
ax = erfp_df[i].plot(kind='kde')

In [56]:
erfp_df[erfp_df.MEAS_CARBON_AMT_LBS_Zscore_abs > 3].count()

In [94]:
fig, ax = plt.subplots(figsize=(16,8))
ax.scatter(erfp_df['MEAS_CARBON_AMT_LBS'], erfp_df['MEAS_CARBON_AMT_LBS_Zscore'])
ax.scatter(erfp_df_1['MEAS_CARBON_AMT_LBS'], erfp_df_1['MEAS_CARBON_AMT_LBS_Zscore'])
ax.set_xlabel('CARBON_AMOUNT')
ax.set_ylabel('z_score')
plt.title("Z-score vs Carbon amount - Carbon amount (scatter plot)") 
plt.show()

In [58]:
# IQR
q1, q3= np.percentile(erfp_df.Log_MEAS_CARBON_AMT_LBS,[25,75])
iqr = q3 - q1

In [59]:
LB = q1 -(1.5 * iqr) 
UB = q3 +(1.5 * iqr) 

In [61]:
LB

In [40]:
i = 'Log_MEAS_CARBON_AMT_LBS'

plt.figure(figsize=(10,8))
plt.subplot(211)
plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
plt.axvline(x=LB, color='r', ls = '--')
plt.axvspan(xmin=LB, xmax=erfp_df[i].min()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.axvline(x=UB, color='r', ls = '--')
plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.title("IQR Outlier detection - Carbon amount (line graph)") 

ax = erfp_df[i].plot(kind='kde')

plt.subplot(212)
plt.xlim(erfp_df[i].min(), erfp_df[i].max()*1.1)
sns.boxplot(x=erfp_df[i])
plt.axvline(x=LB, color='r', ls = '--')
plt.axvspan(xmin=LB, xmax=erfp_df[i].min()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.axvline(x=UB, color='r', ls = '--')
plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
plt.title("IQR Outlier detection - Carbon amount (box plot)") 

In [62]:
i = 'Log_MEAS_CARBON_AMT_LBS'

erfp_df['Outlier'] = 0
 
erfp_df.loc[erfp_df[i] < LB, 'Outlier'] = 1
erfp_df.loc[erfp_df[i] > UB, 'Outlier'] = 1

In [43]:
erfp_df.MEAS_CARBON_AMT_LBS.mad()

In [ ]:
type(erfp_df.MEAS_ENERGY_AMT)

In [29]:
erfp_df.shape

In [79]:
erfp_df[erfp_df.Outlier != 1].MEAS_CARBON_AMT_LBS.max()

In [64]:
erfp_df[erfp_df.Outlier == 1].count()

In [80]:
# OUTLIER
erfp_df[erfp_df.Outlier == 1].to_excel('C:\\Users\\USER\\Documents\\Green_score\\carbon_outliers.xlsx', 
                sheet_name='CARBON_outlier')

In [82]:
import datetime as d
file = "C:\\Users\\USER\\Documents\\Green_score\\CARBON.xlsx" 
with pd.ExcelWriter(file) as writer: 
    erfp_df[erfp_df.Outlier == 1].to_excel(writer, sheet_name='OUTLIER', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.Outlier == 0].to_excel(writer, sheet_name='BENCHMARK', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))